<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 9 · Interest-Rate Swaps

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch09_interest_rate_swaps.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This lab connects Chapter 9 to a reproducible calculation. You will inspect the chapter's inputs, reproduce a core interest-rate swaps relationship, and test how a controlled change affects the result.

By the end, you should be able to identify the model inputs and units, verify one numerical output, and state the limitation that matters before interpreting it.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The next cell runs the transparent Python implementation and prints its reproducible summary. Read the inputs and units in the script alongside the output; the printed values are teaching examples, not investment recommendations.

In [ ]:
SCRIPT = ROOT / "code" / "ch09_interest_rate_swaps.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Use the script output as controlled evidence. Check the units, sign convention, source dataset, and the economic direction of the result before using it in a written explanation or GenAI-assisted summary.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch09_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Recompute the par rate, verify par repricing, and show the receiver/payer DV01 sign convention explicitly.

In [ ]:
swap = pd.read_csv(DATA / "ch09_swap_inputs.csv")
annuity = (swap.alpha * swap.discount).sum()
floating = (swap.alpha * swap.discount * swap.forward).sum()
par_rate = floating / annuity
notional = 50_000_000
receiver_value_at_par = notional * annuity * (par_rate - par_rate)
receiver_dv01 = -notional * annuity * 0.0001
payer_dv01 = -receiver_dv01
pd.DataFrame({
    "check": ["par_rate", "par_repricing", "receiver_dv01", "payer_dv01"],
    "value": [par_rate, receiver_value_at_par, receiver_dv01, payer_dv01],
}).round(6)

## Mini-lab: Reconcile swap legs and the par rate

Project floating coupons from the supplied forward curve and discount them using the separate discount factors. The single-curve identity 1 minus the final discount factor does not apply to these inputs.

In [ ]:
import pandas as pd

swap = pd.read_csv(DATA / "ch09_swap_inputs.csv")
annuity = (swap.alpha * swap.discount).sum()
floating_leg = (swap.alpha * swap.discount * swap.forward).sum()
par_rate = floating_leg / annuity
pd.Series({"annuity": annuity, "floating_leg": floating_leg,
           "par_rate": par_rate})


## Exercises

Use the displayed inputs and the chapter script to complete these checks. Record units and sign conventions with every result.

1. Recompute the par fixed rate from the floating leg and annuity.
2. Value the receive-fixed swap at 4.5% instead of 4.0%.
3. State the payer/receiver sign convention used in the valuation.

For more detail, see Chapter 9 of *Python & AI for Rates, Bonds, and Credit* and its referenced sections on the topic.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">